# Notebook 1: Training and Representation Metrics

Colab-ready workflow for training a flow-circuits encoder on a frozen ResNet18 and inspecting its representation quality.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jacobposchl/model-interpretability.git'
REPO_DIR = Path('/content/model_interpretability' if IN_COLAB else Path.cwd()).resolve()

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/flow_circuits')
    DATA_ROOT  = Path('/content/drive/MyDrive/ctls/data')
else:
    if not (REPO_DIR / 'flow_circuits').exists():
        raise RuntimeError('Run this notebook from the repository root or in Google Colab.')
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'local_notebook_runtime'
    DATA_ROOT  = DRIVE_ROOT / 'data'

BACKBONE_ROOT   = DRIVE_ROOT / 'backbones'
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments'
NOTEBOOK_ROOT   = DRIVE_ROOT / 'notebook_runs'
for path in (DRIVE_ROOT, DATA_ROOT, BACKBONE_ROOT, EXPERIMENT_ROOT, NOTEBOOK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_ROOT = REPO_DIR / 'configs' / 'flow'
print(f"Repo      : {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Data      : {DATA_ROOT}")

In [ ]:
from flow_circuits.data import CIFAR10_STATS, build_cifar10_splits
from flow_circuits.evaluation import evaluate_representation_metrics
from flow_circuits.training import collect_model_outputs, load_components_from_checkpoint

## Step 1 — Configure Your Run

Edit the cell below. Everything else runs automatically.

---

**`TRAINING_MODE`** — controls how long training runs:

| Mode | Epochs | Time on T4 GPU | Use for |
|------|--------|----------------|---------|
| `'smoke'` | 1 per phase | ~2 min | Verifying the pipeline runs end-to-end |
| `'debug'` | 5 per phase | ~30 min | Checking loss curves move in the right direction |
| `'full'` | Config defaults (20+20) | 3–6 hr | Scientifically valid results |

> **Note:** Results from `'smoke'` or `'debug'` are **not** scientifically meaningful — the model has not converged.

---

**`CONFIG_NAME`** — selects the model configuration:

- `'resnet18_base'` — Phases A + B only. No trajectory alignment (Phase C skipped). Faster.
- `'resnet18_aligned'` — Full pipeline with Phase C trajectory alignment sweep.

---

**Path overrides** — set these to reuse files you've already computed. Leave as `None` to auto-generate.

In [ ]:
# ── Training mode ─────────────────────────────────────────────────────────────
#   'smoke' = 1 epoch, ~2 min    (pipeline check only — results not meaningful)
#   'debug' = 5 epochs/phase, ~30 min on GPU  (trends are directionally valid)
#   'full'  = config defaults, 3-6 hr on T4   (scientifically valid results)
TRAINING_MODE = 'smoke'  # <- change me

# ── Model configuration ───────────────────────────────────────────────────────
#   'resnet18_base'    = Phases A+B only (no trajectory alignment)
#   'resnet18_aligned' = Full pipeline with Phase C trajectory alignment sweep
CONFIG_NAME = 'resnet18_aligned'  # <- change me

# ── Reuse existing artifacts (set to None to auto-generate) ───────────────────
BACKBONE_WEIGHTS_PATH = None  # path to a trained ResNet18-CIFAR10 .pt file
CHECKPOINT_PATH       = None  # path to a trained flow-circuits final.pt file

# ── Backbone training (only runs if BACKBONE_WEIGHTS_PATH doesn't exist) ──────
BACKBONE_EPOCHS     = 15
BACKBONE_BATCH_SIZE = 128

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = str(NOTEBOOK_ROOT / 'nb01')

In [ ]:
import copy
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml

# ── Training mode settings ────────────────────────────────────────────────────
_MODE_SETTINGS = {
    'smoke': {
        'label': 'Smoke test (~2 min) — pipeline check only, results NOT meaningful',
        'batch_size': 32, 'num_workers': 0,
        'phase_epochs': {'phase_a': 1, 'phase_b': 1, 'phase_c': 1},
        'baseline_fit': 128, 'baseline_eval': 128, 'val_images': 64,
        'align_pairs': 256, 'disc_images': 256, 'disc_bootstrap': 4, 'interv_images': 128,
    },
    'debug': {
        'label': 'Debug run (~30 min on GPU) — trends meaningful, not publication quality',
        'batch_size': 64, 'num_workers': 2,
        'phase_epochs': {'phase_a': 5, 'phase_b': 5, 'phase_c': 5},
        'baseline_fit': 512, 'baseline_eval': 512, 'val_images': 512,
        'align_pairs': 1024, 'disc_images': 2000, 'disc_bootstrap': 10, 'interv_images': 1000,
    },
    'full': {
        'label': 'Full training (3-6 hr on T4) — scientifically valid results',
        'batch_size': None, 'num_workers': None, 'phase_epochs': None,
        'baseline_fit': None, 'baseline_eval': None, 'val_images': None,
        'align_pairs': None, 'disc_images': None, 'disc_bootstrap': None, 'interv_images': None,
    },
}
_ms = _MODE_SETTINGS[TRAINING_MODE]
print(f"Training mode : {TRAINING_MODE}")
print(f"  {_ms['label']}")

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLI_MODULES = {
    'flow-train-backbone': 'flow_circuits.cli.train_backbone',
    'flow-train': 'flow_circuits.cli.train',
    'flow-evaluate': 'flow_circuits.cli.evaluate',
}

def run_flow_cli(command: str, *args: str) -> None:
    def _stream(cmd):
        import os
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, cwd=REPO_DIR, env=env,
        )
        while True:
            line = process.stdout.readline()
            if not line:
                break
            print(line, end="", flush=True)
        process.wait()
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)
    try:
        _stream([command, *args])
    except FileNotFoundError:
        _stream([sys.executable, '-m', CLI_MODULES[command], *args])

with open(str(CONFIG_ROOT / f'{CONFIG_NAME}.yaml'), 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

if not BACKBONE_WEIGHTS_PATH:
    BACKBONE_WEIGHTS_PATH = str(BACKBONE_ROOT / f"{config['backbone']['arch']}_cifar10_supervised.pt")
config['data']['data_dir'] = str(DATA_ROOT)
config['data']['download'] = False
config['backbone']['weights_path'] = str(BACKBONE_WEIGHTS_PATH)
config['logging']['checkpoint_dir'] = str(EXPERIMENT_ROOT / config['experiment']['name'])

effective_config = copy.deepcopy(config)
effective_phase_epochs = None if _ms['phase_epochs'] is None else copy.deepcopy(_ms['phase_epochs'])
if effective_phase_epochs is not None and effective_config['experiment'].get('mode', 'base') != 'aligned':
    effective_phase_epochs['phase_c'] = 0
if TRAINING_MODE != 'full':
    effective_config['data']['batch_size'] = _ms['batch_size']
    effective_config['data']['num_workers'] = _ms['num_workers']
    effective_config['training']['phase_epochs'] = effective_phase_epochs
    effective_config['training']['baseline_fit_images'] = _ms['baseline_fit']
    effective_config['training']['baseline_eval_images'] = _ms['baseline_eval']
    effective_config['training']['validation_images'] = _ms['val_images']
    effective_config['training']['alignment_max_pairs'] = _ms['align_pairs']
    effective_config['discovery']['max_images'] = _ms['disc_images']
    effective_config['discovery']['bootstrap_iterations'] = _ms['disc_bootstrap']
    effective_config['interventions']['max_images'] = _ms['interv_images']
    effective_config['logging']['checkpoint_dir'] = str(OUTPUT_DIR / f'{TRAINING_MODE}_checkpoints')

EFFECTIVE_CONFIG = OUTPUT_DIR / f'{TRAINING_MODE}_config.yaml'
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(effective_config, sort_keys=False), encoding='utf-8')

PHASE_B_CHECKPOINT = Path(effective_config['logging']['checkpoint_dir']) / 'phase_b.pt'
EFFECTIVE_CHECKPOINT = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'final.pt'
EVALUATION_PATH = OUTPUT_DIR / 'evaluation.json'
print(f"\nConfig      : {EFFECTIVE_CONFIG}")
print(f"Phases      : {effective_config['training']['phase_epochs']}")
print(f"Phase B ckpt: {PHASE_B_CHECKPOINT}")
print(f"Checkpoint  : {EFFECTIVE_CHECKPOINT}")

## Step 2 — Train and Evaluate

Running the cell below will:
1. Train a supervised ResNet18 backbone on CIFAR-10 (if not already done)
2. Train the flow-circuit encoder (Phases A, B, and optionally C)
3. Evaluate the trained model on the held-out test set

Each step is skipped automatically if its output file already exists.

In [ ]:
# ── Step 1: Train backbone (skipped if checkpoint exists) ─────────────────────
if not Path(BACKBONE_WEIGHTS_PATH).exists():
    print("Backbone checkpoint not found — training ResNet18 on CIFAR-10...")
    run_flow_cli(
        'flow-train-backbone',
        '--arch', effective_config['backbone']['arch'],
        '--data-dir', effective_config['data']['data_dir'],
        '--output', str(BACKBONE_WEIGHTS_PATH),
        '--batch-size', str(BACKBONE_BATCH_SIZE),
        '--num-workers', str(effective_config['data'].get('num_workers', 0)),
        '--seed', str(effective_config['data'].get('seed', 0)),
        '--epochs', str(BACKBONE_EPOCHS),
    )
else:
    print(f"Backbone checkpoint found — skipping backbone training.")
    print(f"  {BACKBONE_WEIGHTS_PATH}")

# ── Step 2: Train flow model (skipped if checkpoint exists) ───────────────────
RESUME_CHECKPOINT = PHASE_B_CHECKPOINT if (not EFFECTIVE_CHECKPOINT.exists() and PHASE_B_CHECKPOINT.exists()) else None
if not EFFECTIVE_CHECKPOINT.exists():
    if RESUME_CHECKPOINT is not None:
        print("\nFinal checkpoint not found — resuming from Phase B checkpoint...")
        print(f"  {RESUME_CHECKPOINT}")
        run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG), '--resume', str(RESUME_CHECKPOINT))
    else:
        print("\nFlow model checkpoint not found — starting training...")
        run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG))
else:
    print(f"\nFlow model checkpoint found — skipping training.")
    print(f"  {EFFECTIVE_CHECKPOINT}")

# ── Step 3: Evaluate ──────────────────────────────────────────────────────────
print("\nRunning evaluation on held-out test set...")
run_flow_cli('flow-evaluate', '--checkpoint', str(EFFECTIVE_CHECKPOINT), '--output', str(EVALUATION_PATH))
evaluation_summary = json.loads(EVALUATION_PATH.read_text(encoding='utf-8'))

if TRAINING_MODE == 'smoke':
    print("\n*** SMOKE MODE: Results above are from 1-epoch training. ***")
    print("*** Set TRAINING_MODE='full' for scientifically valid results. ***")

## Step 3 — Inspect Results

The cell below loads the trained model and shows what the learned representations look like spatially.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
components = load_components_from_checkpoint(EFFECTIVE_CHECKPOINT, device=device)
checkpoint_phase = components.checkpoint.get('phase', 'unknown')
phase_c_summary = components.checkpoint.get('summary', {}).get('phase_c')
print(f"Loaded checkpoint : {EFFECTIVE_CHECKPOINT}")
print(f"Saved phase       : {checkpoint_phase}")
print(f"Phase B exists    : {PHASE_B_CHECKPOINT.exists()}")
if phase_c_summary is not None:
    if phase_c_summary.get('accepted'):
        print("Phase C result    : accepted")
        print(
            f"  lambda={phase_c_summary.get('lambda_traj')}  topk={phase_c_summary.get('traj_topk')}"
            f"  gamma={phase_c_summary.get('traj_gamma')}  tau={phase_c_summary.get('traj_tau')}"
        )
    else:
        print(f"Phase C result    : skipped/rejected ({phase_c_summary.get('reason', 'unknown')})")
loaders = build_cifar10_splits(
    data_dir=components.config['data']['data_dir'],
    batch_size=components.config['data']['batch_size'],
    num_workers=components.config['data'].get('num_workers', 0),
    seed=components.config['data'].get('seed', 0),
    augment_fit=components.config['data'].get('augment_fit', True),
    download=components.config['data'].get('download', True),
)
outputs = collect_model_outputs(
    components,
    loaders['test'],
    device=device,
    max_images=16 if TRAINING_MODE == 'smoke' else (256 if TRAINING_MODE == 'debug' else None),
)
metrics = evaluate_representation_metrics(
    outputs['z'],
    outputs['local_features'],
    outputs['flow_targets'],
    outputs['future_descriptors'],
    outputs['predicted_next'],
    outputs['reconstructed_current'],
    max_alignment_pairs=512 if TRAINING_MODE == 'smoke' else 2048,
)

# Print human-readable summary
em = evaluation_summary
rm = em['representation_metrics']
bl = em['baseline_comparison']
cc = em.get('confirmatory_checks', {})
nc = em.get('null_checks', {})

print("=" * 60)
print(f"Results  (n={em['n_images']:,} test images, mode={TRAINING_MODE})")
print("=" * 60)
print(f"\n  Prediction cosine  : {rm['prediction_cosine_mean']:.4f}")
print(f"    Measures how well the encoder predicts next-layer flow.")
print(f"    Best simple baseline: {bl['best_baseline']:.4f}  Ours: {rm['prediction_cosine_mean']:.4f}  (+{rm['prediction_cosine_mean']-bl['best_baseline']:.4f})")
print(f"\n  Reconstruction cos : {rm['reconstruction_cosine_mean']:.4f}")
print(f"    Measures how well the encoder reconstructs current-layer flow.")
print(f"\n  Trajectory align   : {rm['trajectory_alignment_mean']:.4f}")
print(f"    Measures spatial consistency of representations across images.")
if cc:
    p1 = cc.get('p1_prediction_vs_best_baseline', {})
    p2 = cc.get('p2_alignment_vs_best_baseline', {})
    print(f"\n  P1 (prediction > baseline) : {'PASS' if p1.get('passes') else 'FAIL'}")
    print(f"  P2 (alignment > baseline)  : {'PASS' if p2.get('passes') else 'FAIL'}")
if nc:
    print(f"\n  Null check — shuffle future  : drop={nc.get('future_shuffle_prediction',{}).get('drop',0):.4f}  (expect large positive)")
    print(f"  Null check — permute depth   : drop={nc.get('depth_order_alignment',{}).get('drop',0):.4f}  (expect positive)")
if TRAINING_MODE == 'smoke':
    print("\n*** SMOKE MODE: numbers above are not meaningful. Train with 'full' mode. ***")

In [ ]:
mean = torch.tensor(CIFAR10_STATS['mean']).view(3, 1, 1)
std = torch.tensor(CIFAR10_STATS['std']).view(3, 1, 1)
sample_image = (outputs['images'][0].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
sample_label = int(outputs['labels'][0].item())
grid_size = components.tokenizer.grid_size
prediction_map = (outputs['predicted_next'][0, 0] * outputs['flow_targets'][0, 1]).sum(dim=-1).view(grid_size, grid_size).cpu().numpy()
reconstruction_map = (outputs['reconstructed_current'][0, 0] * outputs['flow_targets'][0, 0]).sum(dim=-1).view(grid_size, grid_size).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(sample_image)
axes[0].set_title(f'Sample image (label={sample_label})')
axes[0].axis('off')
pred_im = axes[1].imshow(prediction_map, cmap='viridis')
axes[1].set_title('Layer 1 next-step cosine')
recon_im = axes[2].imshow(reconstruction_map, cmap='magma')
axes[2].set_title('Layer 0 reconstruction cosine')
fig.colorbar(pred_im, ax=axes[1], fraction=0.046, pad=0.04)
fig.colorbar(recon_im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()